### DAS Processing Demo

In [1]:
import numpy as np
from pathlib import Path
from skimage.metrics import structural_similarity as ssim

In [2]:
# Example files
ncf_ref = "20210901_000000_cc_000_conventional.npy"
ncf_v1  = "20210901_000000_cc_000_v1.npy"

disp_ref = "20210901_cc_000_daily_conventional_fv_panel.npy"
disp_v1 = "20210901_cc_000_daily_v1_fv_panel.npy"

pick_ref = "20210901_cc_000_daily_conventional_pick.npy"
pick_v1  = "20210901_cc_000_daily_v1_pick.npy"

# Define paths
ncf_root = Path("..") / "data" / "ncf_raw"
path_ncf_ref = ncf_root / ncf_ref
path_ncf_v1  = ncf_root / ncf_v1

disp_root = Path("..") / "results" / "dispersion" / "daily" / "VS_000"
path_disp_ref= disp_root / disp_ref
path_disp_v1 = disp_root / disp_v1
path_pick_ref= disp_root / pick_ref
path_pick_v1 = disp_root / pick_v1

# Load NCFs
R = np.load(path_ncf_ref)
V = np.load(path_ncf_v1)

# Load Dispersion data
D_ref = np.load(path_disp_ref)
D_v1  = np.load(path_disp_v1)

# Load Picks
P_ref = np.load(path_pick_ref)
P_v1  = np.load(path_pick_v1)

In [3]:
if R.shape != V.shape:
    raise ValueError(f"Shape mismatch: R {R.shape}, V {V.shape}")

R = R.astype(np.float64, copy=False)
V = V.astype(np.float64, copy=False)

In [4]:
if D_ref.shape != D_v1.shape:
    raise ValueError(f"Shape mismatch: D_ref {D_ref.shape}, D_v1 {D_v1.shape}")

In [5]:
if P_ref.shape != P_v1.shape:
    raise ValueError(f"Shape mismatch: P_ref {P_ref.shape}, P_v1 {P_v1.shape}")

#### Correctness on the NCFs

Let $\mathbf{R} \in \mathbb{R}^{N \times T}$ denote the reference NCF and $\mathbf{V} \in \mathbb{R}^{N \times T}$ the NCF computed using the optimized method, 

where $N$ is the number of traces and $T$ the number of lag samples.


##### 1. Relative Frobenius 
The relative Frobenius error measures the global energy mismatch between the two NCFs:

$$
\varepsilon_F = \frac{\lVert \mathbf{V} - \mathbf{R} \rVert_F} {\lVert \mathbf{R} \rVert_F}
$$
where the Frobenius norm is defined as 
$$
\lVert \mathbf{A} \rVert_F
=
\left(
\sum_{i=1}^{N}
\sum_{j=1}^{T}
A_{ij}^2
\right)^{1/2}.
$$
This metric captures the accumulated energy error across all traces and lags and provides a scale-independent measure of global numerical accuracy. 

In [6]:
diff = V - R

fro_R = np.linalg.norm(R, ord='fro')
fro_diff = np.linalg.norm(diff, ord='fro')

ref_fro_error = fro_diff / fro_R

print("=== Frobenius Norm Comparison ===")
print(f"NCF shape               : {R.shape}")
print(f"||R||_F                 : {fro_R:.6e}")
print(f"||V - R||_F             : {fro_diff:.6e}")
print(f"Relative Frobenius error: {ref_fro_error:.6e}")

=== Frobenius Norm Comparison ===
NCF shape               : (350, 1001)
||R||_F                 : 8.175275e+04
||V - R||_F             : 1.404859e-02
Relative Frobenius error: 1.718424e-07


##### 2. Maximum Absolute Error
The maximum absolute error quantifies the worst-case pointwise deviation:

$$
\varepsilon_{\max}
=
\max_{i,j}
\left|
V_{ij} - R_{ij}
\right|.
$$
This metric is sensitive to localized artifacts.

In [7]:
diff = V - R

max_abs_error = np.max(np.abs(diff))

# Location of worst error
idx_trace, idx_lag = np.unravel_index(np.argmax(np.abs(diff)), diff.shape)

print("=== Max Absolute Error Comparison ===")
print(f"NCF shape           : {R.shape}")
print(f"Max absolute error  : {max_abs_error:.6e}")

=== Max Absolute Error Comparison ===
NCF shape           : (350, 1001)
Max absolute error  : 9.765625e-04


##### 3. Cosine Similarity per Trace
To assess waveform shape agreement independent of amplitude scaling, we compute the cosine similarity for each trace:

$$
\mathrm{cos\_sim}_i
=
\frac{
\mathbf{V}_i \cdot \mathbf{R}_i
}{
\lVert \mathbf{V}_i \rVert_2
\,
\lVert \mathbf{R}_i \rVert_2
},
\qquad i = 1, \ldots, N,
$$
where $\mathbf{V}_i, \mathbf{R}_i \in \mathbb{R}^T$ denote the $i$-th traces of the optimized and reference NCFs, respectively. Cosine similarity values close to unity indicate strong waveform agreement and preserved phase information, which is critical for reliable dispersion-curve extraction.


In [8]:
eps = 1e-15 # avoid divide by zero

dot = np.sum(V * R, axis=1)
nr = np.linalg.norm(R, axis=1)
nv = np.linalg.norm(V, axis=1)

cos_sim = dot / (nr * nv + eps)

# Handle traces that are all zero in both arrays
zero_mask = (nr < eps) & (nv < eps)
cos_sim[zero_mask] = 1.0

print("=== Cosine Similarity per Trace ===")
print(f"NCF shape                 : {R.shape}")
print(f"Mean cosine similarity    : {cos_sim.mean():.6f}")
print(f"Median cosine similarity  : {np.median(cos_sim):.6f}")
print(f"Min cosine similarity     : {cos_sim.min():.6f}")
print(f"5th percentile            : {np.percentile(cos_sim, 5):.6f}")
print(f"95th percentile           : {np.percentile(cos_sim, 95):.6f}")

=== Cosine Similarity per Trace ===
NCF shape                 : (350, 1001)
Mean cosine similarity    : 1.000000
Median cosine similarity  : 1.000000
Min cosine similarity     : 1.000000
5th percentile            : 1.000000
95th percentile           : 1.000000


#### Physics of the NCFs

##### 1. Spectral Comparison Metrics
Let the $i$-th noise cross-correlation trace be denoted by
$R_i[t]$ for the reference (conventional) method and
$V_i[t]$ for the optimized (v1) method, where
$t = 0, \ldots, T-1$ indexes the discrete lag samples.

The discrete Fourier transform is applied along the lag axis:
$$
\hat{R}_i[f] = \mathrm{FFT}\{R_i[t]\}, \qquad
\hat{V}_i[f] = \mathrm{FFT}\{V_i[t]\},
$$
where $f$ denotes the discrete frequency index.

The corresponding amplitude spectra are defined as
$$
A_i^{R}[f] = \left| \hat{R}_i[f] \right|, \qquad
A_i^{V}[f] = \left| \hat{V}_i[f] \right|.
$$

Relative spectral amplitude error is given by:
$$
\varepsilon_\text{spec} = \frac{\lVert A^V - A^R \rVert_F} {\lVert A^R \rVert_F}
$$


In [9]:
N, T = R.shape

# Frequency axis 
dt = 0.004
freq = np.fft.rfftfreq(T, d=dt)

# Amplitude spectra (rFFT)
AR = np.abs(np.fft.rfft(R, axis=1))
AV = np.abs(np.fft.rfft(V, axis=1))

# Band mask: [f1, f2]; refer to cc.yaml
f1, f2 = 1.0, 10.0
band = (freq >= f1) & (freq <= f2)
oob = ~band

eps = 1e-15

# 1. Band-limited relative spectral amplitude error
spec_err_band = np.linalg.norm((AV - AR)[:, band], ord='fro') / (np.linalg.norm(AR[:, band], ord='fro') + eps)

print("=== Band-limited spectral error (1–10 Hz) ===")
print(f"Global rel spectral error    : {spec_err_band:.6e}")

# 2. Leakage ratio: out-of-band energy / in-band energy
leak_R = np.linalg.norm(AR[:, oob], ord='fro') / (np.linalg.norm(AR[:, band], ord='fro') + eps)
leak_V = np.linalg.norm(AV[:, oob], ord='fro') / (np.linalg.norm(AV[:, band], ord='fro') + eps)

print("\n=== Spectral leakage (out-of-band / in-band) ===")
print(f"Leakage ratio (reference)    : {leak_R:.6e}")
print(f"Leakage ratio (v1)           : {leak_V:.6e}")
print(f"Leakage ratio difference     : {(leak_V - leak_R):.6e}")

=== Band-limited spectral error (1–10 Hz) ===
Global rel spectral error    : 7.940710e-08

=== Spectral leakage (out-of-band / in-band) ===
Leakage ratio (reference)    : 2.094905e-01
Leakage ratio (v1)           : 2.094905e-01
Leakage ratio difference     : -1.855503e-09


##### 2. Strutural Similarity Index (SSIM) for Dispersion Images

We further assess whether downstream physical interpretations are preserved. In particular, we compare dispersion images and retrieved phase-velocity curves, which are nonlinear transforms of the NCFs and therefore more sensitive to numerical perturbations.

Let $D_{\text{conv}}(f,c)$ and $D_{\text{v1}}(f,c)$ denote dispersion images obtained from the conventional and v1 cross-correlation methods, respectively, where $f$ is frequency and $c$ is phase velocity. 

To quantify the similarity between dispersion images, we use the Structural Similarity Index (SSIM), which measures similarity in terms of luminance, contrast, and structural information. SSIM is defined as
$$
\mathrm{SSIM}(X,Y) =
\frac{(2\mu_X\mu_Y + C_1)(2\sigma_{XY} + C_2)}
     {(\mu_X^2 + \mu_Y^2 + C_1)(\sigma_X^2 + \sigma_Y^2 + C_2)},
$$

where $\mu_X, \mu_Y$ are mean intensities, $\sigma_X^2, \sigma_Y^2$ are variances, $\sigma_{XY}$ is the covariance, and $C_1, C_2$ are small stabilizing constants.

SSIM values range from $-1$ to $1$, with values close to $1$ indicating near-identical structural content. In this context, SSIM $\approx 1$ implies that the v1 method preserves the same dispersion ridges and modal structure as the conventional method.

In [10]:
# Normalize first
scale = np.percentile(np.abs(D_ref), 99.5)

D_ref_n = np.clip(D_ref, -scale, scale) / scale
D_v1_n  = np.clip(D_v1,  -scale, scale) / scale

In [11]:
ssim_val = ssim(
    D_ref_n,
    D_v1_n,
    data_range=2.0,
    gaussian_weights=True,
    sigma=1.5,
    use_sample_covariance=False
)

In [12]:
ssim_val

np.float64(1.0000000019651794)

##### 3. Phase-Velocity Curve Comparison

Dispersion images are intermediate products; the final quantity of interest is the frequency-dependent phase-velocity curve $c(f)$. Let $c_{\text{conv}}(f)$ and $c_{\text{v1}}(f)$ denote phase-velocity curves picked from dispersion images generated using the conventional and v1 methods, respectively.

We define the phase-velocity difference as
$$
\Delta c(f) = c_{\text{v1}}(f) - c_{\text{conv}}(f),
$$
and evaluate its magnitude using summary statistics:
$$
\text{Median}(|\Delta c|), \quad
\text{RMS}(|\Delta c|), \quad
P_{95}(|\Delta c|).
$$

In [13]:
print("P_ref shape:", P_ref.shape, "dtype:", P_ref.dtype)
print("P_v1  shape:", P_v1.shape,  "dtype:", P_v1.dtype)

P_ref shape: (204,) dtype: float32
P_v1  shape: (204,) dtype: float32


In [14]:
# Mask invalid values
P_ref = P_ref.astype(np.float64, copy=False)
P_v1  = P_v1.astype(np.float64, copy=False)

mask = np.isfinite(P_ref) & np.isfinite(P_v1)

In [15]:
# Phase-velocity difference
dc = P_v1[mask] - P_ref[mask]

# Error metrics
pick_err = {
    "n_freq": int(mask.sum()),

    # absolute errors (m/s or km/s depending on units)
    "median_abs_dc": float(np.median(np.abs(dc))),
    "rms_abs_dc": float(np.sqrt(np.mean(dc**2))),
    "p95_abs_dc": float(np.percentile(np.abs(dc), 95)),
    "max_abs_dc": float(np.max(np.abs(dc))),

    # relative (dimensionless)
    "median_rel": float(np.median(np.abs(dc / P_ref[mask]))),
    "p95_rel": float(np.percentile(np.abs(dc / P_ref[mask]), 95)),
}

In [16]:
pick_err

{'n_freq': 204,
 'median_abs_dc': 0.0,
 'rms_abs_dc': 0.0,
 'p95_abs_dc': 0.0,
 'max_abs_dc': 0.0,
 'median_rel': 0.0,
 'p95_rel': 0.0}